# 1.1 - Data Wrangling

The purpose of this notebook is to transform the data to obtain training labels from the `outcome` column of the data obtained from LL2 API, GCAT, and Course-provided launch data. Additionally, data from wikipedia will be transformed for later processing.

In [4]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
from pathlib import Path

In [5]:
csv_path = Path.cwd().parent / 'data' / 'interim' / 'api-launch-data-table.csv'
launch_df = pd.read_csv(csv_path)

In [6]:
launch_df.head(10)

,launch_designator,booster_version,orbit,launch_site,flights,reused,landing_pad,block,serial,longitude,latitude,reused_count,outcome,OrbPay,launch_date,gridfins,legs
0,2020-084,Falcon 9,Low Earth Orbit,Launch Complex 39A,23,False,JRTI,Block 5,B1061,-80.604282,28.608227,0,True ASDS,13.000,2020-11-16,True,True
1,2020-078,Falcon 9,Medium Earth Orbit,Space Launch Complex 40,23,False,OCISLY,Block 5,B1062,-80.577357,28.561941,0,True ASDS,4.400,2020-11-05,True,True
2,2020-074,Falcon 9,Low Earth Orbit,Space Launch Complex 40,20,True,JRTI,Block 5,B1060,-80.577357,28.561941,2,True ASDS,16.000,2020-10-24,True,True
3,2020-073,Falcon 9,Low Earth Orbit,Launch Complex 39A,14,True,OCISLY,Block 5,B1051,-80.604282,28.608227,5,True ASDS,16.000,2020-10-18,True,True
4,2020-070,Falcon 9,Low Earth Orbit,Launch Complex 39A,19,True,OCISLY,Block 5,B1058,-80.604282,28.608227,2,True ASDS,16.000,2020-10-06,True,True
5,2020-062,Falcon 9,Low Earth Orbit,Launch Complex 39A,20,True,OCISLY,Block 5,B1060,-80.604282,28.608227,1,True ASDS,16.000,2020-09-03,True,True
6,2020-059,Falcon 9,Sun-Synchronous Orbit,Space Launch Complex 40,6,True,LZ-1,Block 5,B1059,-80.577357,28.561941,3,True RTLS,3.091,2020-08-30,True,True
7,2020-057,Falcon 9,Low Earth Orbit,Space Launch Complex 40,11,True,OCISLY,Block 5,B1049,-80.577357,28.561941,5,True ASDS,15.815,2020-08-18,True,True
8,2020-055,Falcon 9,Low Earth Orbit,Launch Complex 39A,14,True,OCISLY,Block 5,B1051,-80.604282,28.608227,4,True ASDS,15.335,2020-08-07,True,True
9,2020-048,Falcon 9,Geostationary Transfer Orbit,Space Launch Complex 40,19,True,JRTI,Block 5,B1058,-80.577357,28.561941,1,True ASDS,5.600,2020-07-20,True,True


identify and calculate the percentage of missing values from each column.

In [7]:
print('{}'.format(np.round(100*launch_df[~launch_df.isnull()].count()/launch_df.shape[0],2)))

launch_designator    100.00
booster_version      100.00
orbit                 98.96
launch_site          100.00
flights              100.00
reused               100.00
landing_pad          100.00
block                100.00
serial               100.00
longitude            100.00
latitude             100.00
reused_count         100.00
outcome              100.00
OrbPay               100.00
launch_date          100.00
gridfins             100.00
legs                 100.00
dtype: float64


Identify which columns are numerical or categorical

In [8]:
launch_df.dtypes

launch_designator        str
booster_version          str
orbit                    str
launch_site              str
flights                int64
reused                  bool
landing_pad              str
block                    str
serial                   str
longitude            float64
latitude             float64
reused_count           int64
outcome                  str
OrbPay               float64
launch_date              str
gridfins                bool
legs                    bool
dtype: object

Calculate the number of launches on each site

In [9]:
launch_df['launch_site'].value_counts()

launch_site
Space Launch Complex 40    58
Launch Complex 39A         23
Space Launch Complex 4E    15
Name: count, dtype: int64

Calculate the number and occurences of each orbit

In [10]:
launch_df['orbit'].value_counts()

orbit
Low Earth Orbit                 54
Geostationary Transfer Orbit    28
Sun-Synchronous Orbit            6
Medium Earth Orbit               3
Lunar Orbit                      1
Polar Orbit                      1
High Earth Orbit                 1
Heliocentric L1                  1
Name: count, dtype: int64

Calculate the number and occurence of mission outcome

In [11]:
landing_outcomes = launch_df['outcome'].value_counts()
print(landing_outcomes)

outcome
True ASDS      44
None EXP       20
True RTLS      14
False ASDS      7
None Ocean      6
False Ocean     2
False PCL       2
False RTLS      1
Name: count, dtype: int64


Create a set of bad outcomes -- landing outcomes with False or None.

In [12]:
bad_outcomes = set()
for outcome in landing_outcomes.keys():
    if 'False' in outcome or 'None' in outcome:
        bad_outcomes.add(outcome)

In [13]:
bad_outcomes

{'False ASDS',
 'False Ocean',
 'False PCL',
 'False RTLS',
 'None EXP',
 'None Ocean'}

Create a landing outcome label from `outcome` column.

Here a script is created to create the `Class` column (the target variable for analysis) to the launch data from the specified csv path to the launches DataFrame.

In [14]:
def add_class(launch_csv: str|Path, save_path: None|str|Path = None) -> None|Path:
    """
    Function that adds the target 'Class' column to the launch data from the specified launch csv path,
    and saves the result to the specified save path.

    :param launch_csv: str or Path; file path to the launches csv file
    :param save_path: str or Path; Default None; file path where the result is to be saved. If None,
                      result will be saved in the current working directory.
    :return save_path: Path; file path where result is saved. 
    """
    try:
        launch_df = pd.read_csv(launch_csv)
        
    except Exception as e:
        print(e)
        return None

    # Drop NA Values
    launch_df.dropna(axis=0, inplace=True)
    
    landing_outcomes = launch_df['outcome'].unique()

    bad_outcomes = set()
    for outcome in landing_outcomes:
        if 'False' in outcome or 'None' in outcome:
            bad_outcomes.add(outcome)
            
    def outcome_map(entry: str) -> int:
        """
        Utility function that maps an outcome to either the integer 1 or 0.
        If the outcome is a landing success, the function returns 1; otherwise,
        the function returns 0

        :param entry: str; a landing outcome in the format "(Landing Success) (Landing Pad)", e.g., True ASDS.
        :return outcome: int; if the entry is a landing success, `outcome=1`; otherwise, `outcome=0`.
        """

        if entry in bad_outcomes:
            outcome = 0
        else:
            outcome = 1

        return outcome

    launch_df['class'] = launch_df['outcome'].map(outcome_map)

    if not save_path:
        save_path = Path.cwd() / 'api-launch-data-table-class.csv'

    if isinstance(save_path, str):
        save_path = Path(save_path)
        
    launch_df.to_csv(save_path, index=False)

    return save_path

In [15]:
# Test the script
from spacey_falcon_9_project.dataset import add_class

launch_csv = Path.cwd().parent / 'data' / 'interim' / 'api-launch-data-table.csv'
save_path = Path.cwd().parent / 'data' / 'processed' / 'api-launch-data-table-class.csv'

add_class(launch_csv, save_path)

PosixPath('/Users/jelo/spacey_falcon_9_project/data/processed/api-launch-data-table-class.csv')

In [16]:
launches_with_class = pd.read_csv(save_path)
launches_with_class[['outcome', 'class']]

,outcome,class
0,True ASDS,1
1,True ASDS,1
2,True ASDS,1
3,True ASDS,1
4,True ASDS,1
...,...,...
90,None EXP,0
91,None EXP,0
92,None EXP,0
93,False PCL,0


## Wikipedia Launch Data

In [17]:
file_path = Path.cwd().parent / 'data/interim/wikipedia-launch-data-table.csv'
df = pd.read_csv(file_path)
df.head()

,Flight No.,Date andtime (UTC),"Version,Booster [b]",Launch site,Payload[c],Payload mass,Orbit,Customer,Launchoutcome,Boosterlanding,Description
0,1,"4 June 2010,18:45",F9 v1.0B0003.1,"CCAFS,SLC-40",Dragon Spacecraft Qualification Unit,NaN,LEO,SpaceX,Success,Failure(parachute),First flight of Falcon 9 v1.0. Used a boilerpl...
1,2,"8 December 2010,15:43",F9 v1.0B0004.1,"CCAFS,SLC-40",Dragon demo flight C1(Dragon C101),NaN,LEO (ISS),\nNASA (COTS)\nNRO,Success,Failure(parachute),"Maiden flight of Dragon capsule, consisting of..."
2,3,"22 May 2012,07:44",F9 v1.0B0005.1,"CCAFS,SLC-40",Dragon demo flight C2+(Dragon C102),"525 kg (1,157 lb)",LEO (ISS),NASA (COTS),Success,No attempt,Dragon spacecraft demonstrated a series of tes...
3,4,"8 October 2012,00:35",F9 v1.0B0006.1,"CCAFS,SLC-40",SpaceX CRS-1(Dragon C103),"4,700 kg (10,400 lb)",LEO (ISS),NASA (CRS),Success,No attempt,"CRS-1 was successful, but the secondary payloa..."
4,5,"1 March 2013,15:10",F9 v1.0B0007.1,"CCAFS,SLC-40",SpaceX CRS-2(Dragon C104),"4,877 kg (10,752 lb)",LEO (ISS),NASA (CRS),Success,No attempt,Last launch of the original Falcon 9 v1.0 laun...


In [18]:
df['Launch site'].value_counts()

Launch site
CCAFS,SLC-40             39
KSC,LC-39A               33
Cape Canaveral,LC-40     20
VAFB,SLC-4E              16
CCSFS,SLC-40             12
Cape Canaveral,SLC-40     1
CCAFSSLC-40               1
Name: count, dtype: int64

`CCAFS,SLC-40`, `Cape Canaveral,LC-40`, `CCSFS,SLC-40`, `Cape Canaveral,SLC-40`, and `CCAFSSLC-40` are names of the same launch site: Space Launch Complex 40 -- previously named as LC 40. For consistency, these names were changed to `SLC-40`.

In [19]:
df['Launch site'].unique()

<StringArray>
[         'CCAFS,SLC-40',           'VAFB,SLC-4E',  'Cape Canaveral,LC-40',
            'KSC,LC-39A', 'Cape Canaveral,SLC-40',           'CCAFSSLC-40',
          'CCSFS,SLC-40']
Length: 7, dtype: str

In [20]:
def rename_site(site):
    if site in ['CCAFS,SLC-40', 'Cape Canaveral,LC-40', 'Cape Canaveral,SLC-40', 'CCAFSSLC-40', 'CCSFS,SLC-40']:
        return 'SLC-40'
    else:
        return site

df['Launch site'] = df['Launch site'].map(rename_site)
df['Launch site'].value_counts()

Launch site
SLC-40         73
KSC,LC-39A     33
VAFB,SLC-4E    16
Name: count, dtype: int64

To create the class column, a mission is considered successful if a landing attempt was made and was successful; otherwise, the mission is considered failed. Missions that did not attempt booster landing will be dropped as they were neither successful nor failed.

In [21]:
df['Boosterlanding'].value_counts()

Boosterlanding
Success(drone ship)      64
No attempt               19
Success(ground pad)      16
Failure(drone ship)       6
Controlled(ocean)         4
No attempt                3
Failure(parachute)        2
Uncontrolled(ocean)       2
Failure (drone ship)      2
Precluded(drone ship)     2
Controlled(ocean)         1
Failure(ground pad)       1
Name: count, dtype: int64

In [22]:
landing_outcomes = df['Boosterlanding'].unique()
no_attempt = landing_outcomes[1:3]
mask = df['Boosterlanding'].map(lambda x: not x in no_attempt)
df = df[mask]

In [23]:
successful = landing_outcomes[-3:-1]
def add_class_wiki(x):
    if x in successful:
        return 1
    else:
        return 0

df['class'] = df['Boosterlanding'].map(add_class_wiki)
df['class'].value_counts()

class
1    80
0    20
Name: count, dtype: int64

Next, the `Version,Booster` column was split into two columns as it contains information of the version ID and booster ID .

In [24]:
df['Version,Booster [b]'].unique()

<StringArray>
[       'F9 v1.0B0003.1',        'F9 v1.0B0004.1',          'F9 v1.1B1003',
               'F9 v1.1',          'F9 v1.1B1010',          'F9 v1.1B1012',
          'F9 v1.1B1013',          'F9 v1.1B1015',          'F9 v1.1B1018',
          'F9 FTB1019.1',          'F9 v1.1B1017',          'F9 FTB1020.1',
          'F9 FTB1021.1',          'F9 FTB1022.1',          'F9 FTB1023.1',
          'F9 FTB1024.1',          'F9 FTB1025.1',          'F9 FTB1026.1',
         'F9 FT B1028.1',          'F9 FTB1029.1',          'F9 FTB1031.1',
        'F9 FT ♺B1021.2',          'F9 FTB1032.1',          'F9 FTB1035.1',
       'F9 FT ♺ B1029.2',          'F9 FTB1036.1',          'F9 B4B1039.1',
          'F9 FTB1038.1',          'F9 B4B1040.1',          'F9 B4B1041.1',
       'F9 FT ♺ B1031.2',          'F9 B4B1042.1',       'F9 FT ♺ B1035.2',
       'F9 FT ♺ B1036.2',          'F9 B4B1043.1',       'F9 FT ♺ B1032.2',
          'F9 B4B1045.1',          'F9 B5B1046.1',          'F9 B5B1047.1'

In [25]:
import re

version_pattern = r'F9\s+(v1\.0|v1\.1|FT|B4|B5)'
booster_pattern = r'B\d{4}(?:\.\d+)?'
version_prog = re.compile(version_pattern)
booster_prog = re.compile(booster_pattern)

def get_version_id(entry):
    match = version_prog.match(entry)
    return match.group(1) if match else None

def get_booster_id(entry):
    match = booster_prog.search(entry)
    return match.group(0) if match else None

df['version'] = df['Version,Booster [b]'].map(get_version_id)
df['booster'] = df['Version,Booster [b]'].map(get_booster_id)

In [26]:
df[['Version,Booster [b]', 'version', 'booster']]

,"Version,Booster [b]",version,booster
0,F9 v1.0B0003.1,v1.0,B0003.1
1,F9 v1.0B0004.1,v1.0,B0004.1
5,F9 v1.1B1003,v1.1,B1003
8,F9 v1.1,v1.1,NaN
9,F9 v1.1,v1.1,NaN
...,...,...,...
117,F9 B5 ♺B1051.10,B5,B1051.10
118,F9 B5 ♺B1058.8,B5,B1058.8
119,F9 B5 ♺B1063.2,B5,B1063.2
120,F9 B5 B1067.1,B5,B1067.1


In [27]:
for index, record in df[['Description']].head().iterrows():
    print('Row {}'.format(index))
    print(record.values[0])
    print('')

Row 0
First flight of Falcon 9 v1.0. Used a boilerplate version of Dragon capsule which was not designed to separate from the second stage.(more details below) Attempted to recover the first stage by parachuting it into the ocean, but it burned up on reentry, before the parachutes even deployed.

Row 1
Maiden flight of Dragon capsule, consisting of over 3 hours of testing thruster maneuvering and reentry. Attempted to recover the first stage by parachuting it into the ocean, but it disintegrated upon reentry, before the parachutes were deployed. (more details below) It also included two CubeSats, and a wheel of Brouère cheese.

Row 5
First commercial mission with a private customer, first launch from Vandenberg, and demonstration flight of Falcon 9 v1.1 with an improved 13-tonne to LEO capacity. After separation from the second stage carrying Canadian commercial and scientific satellites, the first stage booster performed a controlled reentry, and an ocean touchdown test for the first 

**Conclusion.** The Wikipedia Launch Data does not seem to add any information to the launch data gathered from LL2 API, GCAT, and the Course-provided JSON. Therefore, it will just be used in certain tasks in the project other than the main tasks.